# Introduction to Structured Generation with Outlines

In this notebook, we'll explore structured generation using the Outlines library, which allows us to constrain LLM outputs to follow specific formats like JSON schemas.

## 1. Installation and Setup

Let's start by installing the required libraries.

In [ ]:
# Install required packages
!pip install outlines transformers torch accelerate pydantic

In [ ]:
# Import necessary libraries
import outlines
import json
from pydantic import BaseModel, Field
from typing import List, Literal, Optional
import time
import re

## 2. Initialize the Model

We'll use a smaller model for this demonstration to ensure it runs efficiently.

In [ ]:
# Initialize the model with Outlines
# Using a smaller model for demonstration - you can use larger models if you have the resources
model = outlines.models.transformers("microsoft/DialoGPT-medium")

print("Model loaded successfully!")

## 3. Traditional (Unconstrained) Generation

Let's first see how traditional LLM generation works without any constraints.

In [ ]:
# Traditional generation without constraints
prompt = "Generate a portfolio summary for client ABC-123 with their asset allocation:"

# Create a simple generator
generator = outlines.generate.text(model)

# Generate response
start_time = time.time()
response = generator(prompt, max_tokens=200)
generation_time = time.time() - start_time

print("Traditional Generation:")
print("-" * 50)
print(response)
print(f"\nGeneration time: {generation_time:.2f} seconds")

## 4. Problems with Traditional Generation

Let's analyze the issues with the traditional approach:

In [ ]:
# Let's try to parse the traditional output
def try_parse_traditional_output(text):
    """Attempt to extract structured data from free-form text"""
    
    print("Attempting to parse traditional output...")
    
    # Try to find client ID
    client_match = re.search(r'client[:\s]+(\w+-\w+)', text, re.IGNORECASE)
    client_id = client_match.group(1) if client_match else "Not found"
    
    # Try to find monetary amounts
    money_pattern = r'\$([\d,]+(?:\.\d{2})?)|([\d,]+(?:\.\d{2})?)\s*(?:dollars|usd|\$)'
    amounts = re.findall(money_pattern, text, re.IGNORECASE)
    
    # Try to find asset types
    asset_types = []
    for asset in ['stock', 'bond', 'cash', 'equity', 'fixed income']:
        if asset in text.lower():
            asset_types.append(asset)
    
    print(f"Extracted Client ID: {client_id}")
    print(f"Found amounts: {amounts}")
    print(f"Detected asset types: {asset_types}")
    
    return {
        "parseable": bool(client_match and amounts),
        "client_id": client_id,
        "amounts": amounts,
        "asset_types": asset_types
    }

# Try to parse our traditional output
parse_result = try_parse_traditional_output(response)
print(f"\nIs output easily parseable? {parse_result['parseable']}")

## 5. Defining a JSON Schema

Now let's define a proper JSON schema for portfolio data using Pydantic:

In [ ]:
# Define our portfolio schema using Pydantic
class Asset(BaseModel):
    asset_type: Literal["stocks", "bonds", "cash", "real_estate", "commodities"]
    value: float = Field(ge=0, description="Asset value in USD")
    percentage: float = Field(ge=0, le=100, description="Percentage of total portfolio")

class Portfolio(BaseModel):
    client_id: str = Field(pattern=r"^[A-Z]{3}-\d{3}$", description="Client identifier")
    total_value: float = Field(ge=0, description="Total portfolio value in USD")
    risk_level: Literal["conservative", "moderate", "aggressive"]
    assets: List[Asset] = Field(min_items=1, max_items=10)
    last_updated: str = Field(pattern=r"^\d{4}-\d{2}-\d{2}$", description="Date in YYYY-MM-DD format")

# Display the schema
schema = Portfolio.model_json_schema()
print("Portfolio JSON Schema:")
print(json.dumps(schema, indent=2))

## 6. Structured Generation with Outlines

Now let's use Outlines to generate portfolio data that conforms to our schema:

In [ ]:
# Create a structured generator using our Pydantic model
structured_generator = outlines.generate.json(model, Portfolio)

# Same prompt but now with structure constraints
structured_prompt = """
Generate a portfolio summary for client ABC-123. The client has a moderate risk profile 
with a diversified portfolio including stocks, bonds, and some cash holdings. 
Total portfolio value is around $750,000.
"""

# Generate structured response
start_time = time.time()
structured_response = structured_generator(structured_prompt)
structured_time = time.time() - start_time

print("Structured Generation Result:")
print("-" * 50)
print(json.dumps(structured_response, indent=2))
print(f"\nGeneration time: {structured_time:.2f} seconds")

## 7. Validation and Benefits

Let's validate our structured output and compare it with the traditional approach:

In [ ]:
# Validate the structured output
try:
    # The output is already validated by Pydantic, but let's do additional checks
    portfolio = Portfolio(**structured_response)
    
    # Check if percentages add up to 100%
    total_percentage = sum(asset.percentage for asset in portfolio.assets)
    
    # Check if asset values add up to total value
    total_asset_value = sum(asset.value for asset in portfolio.assets)
    
    print("Validation Results:")
    print(f"✓ Schema validation: PASSED")
    print(f"✓ Client ID format: {portfolio.client_id} (matches pattern)")
    print(f"✓ Total percentage: {total_percentage:.1f}% (should be ~100%)")
    print(f"✓ Asset values sum: ${total_asset_value:,.2f}")
    print(f"✓ Declared total: ${portfolio.total_value:,.2f}")
    print(f"✓ Value consistency: {abs(total_asset_value - portfolio.total_value) < 1000}")
    
except Exception as e:
    print(f"Validation failed: {e}")

## 8. Comparison Summary

Let's summarize the differences between traditional and structured generation:

In [ ]:
# Create a comparison table
def create_comparison():
    comparison = {
        "Aspect": [
            "Output Format",
            "Parseability", 
            "Data Validation",
            "Consistency",
            "Integration Ready",
            "Generation Time",
            "Error Handling"
        ],
        "Traditional Generation": [
            "Free-form text",
            "Requires complex parsing",
            "Manual validation needed",
            "Variable across runs",
            "Requires post-processing",
            f"{generation_time:.2f}s",
            "Prone to parsing errors"
        ],
        "Structured Generation": [
            "Valid JSON",
            "Directly parseable",
            "Automatic schema validation",
            "Guaranteed structure",
            "Ready for APIs/databases",
            f"{structured_time:.2f}s",
            "Schema prevents errors"
        ]
    }
    
    # Print comparison table
    print("\nComparison: Traditional vs Structured Generation")
    print("=" * 80)
    
    for i, aspect in enumerate(comparison["Aspect"]):
        print(f"{aspect:<20} | {comparison['Traditional Generation'][i]:<25} | {comparison['Structured Generation'][i]}")

create_comparison()

## 9. Key Takeaways

From this introduction, we can see that structured generation offers several advantages:

1. **Reliability**: Guaranteed valid output format
2. **Integration**: Direct compatibility with APIs and databases
3. **Validation**: Automatic constraint checking
4. **Consistency**: Same structure across all generations
5. **Efficiency**: No need for post-processing or error handling

In the next notebook, we'll explore more complex schemas and advanced constraint techniques.

## Exercise

Try modifying the Portfolio schema to include additional fields like:
- `currency` (limited to specific currency codes)
- `performance_metrics` (nested object with returns, volatility, etc.)
- `rebalancing_date` (future date constraint)

Then generate a few examples and observe how the constraints are enforced.